# Обучение и проверка (Colab, GPU)

Тонкий блокнот: весь код в пакете `anemia/` и в `scripts/` репозитория. Запускайте ячейки по порядку.

Перед первым запуском положите на Google Диск архив CP-AnemiC: `MyDrive/anemia/CP-AnemiC_dataset.zip`
(он лежит в общей папке проекта: `md/history/097_2026-10-02_dataset_cp_anemic/`).
Данные и модели хранятся на Диске в `MyDrive/anemia/`, в репозиторий не попадают.

In [ ]:
#@title Ячейка 1 — репозиторий, зависимости, Google Диск
import os, subprocess, sys
REPO = 'zhansaya-afk/jjghg'
BRANCH = 'main'
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/anemia'
os.makedirs(BASE, exist_ok=True)
os.environ['ANEMIA_DATA'] = f'{BASE}/data'
os.environ['ANEMIA_MODELS'] = f'{BASE}/models'

# токен нужен, только если репозиторий приватный: Colab → ключ слева → Secrets → GITHUB_TOKEN
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if os.path.isdir('/content/repo/.git'):
    subprocess.run(['git', '-C', '/content/repo', 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/repo', 'reset', '-q', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, '/content/repo'], check=True)
os.chdir('/content/repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data,dev]'], check=True)
print('Коммит:', subprocess.run(['git', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())

In [ ]:
#@title Ячейка 2 — данные: скачать (один раз) и собрать manifest.csv
import os
D = os.environ['ANEMIA_DATA']
if not os.path.exists(f'{D}/eyes_defy_anemia/labels.csv'):
    !python scripts/fetch_data.py --eyes-defy
cp_zip = '/content/drive/MyDrive/anemia/CP-AnemiC_dataset.zip'
if not os.path.exists(f'{D}/cp_anemic/labels.csv'):
    if os.path.exists(cp_zip):
        !python scripts/fetch_data.py --cp-zip "{cp_zip}"
    else:
        print('⚠️ Нет', cp_zip, '— CP-AnemiC пропущен')
!python scripts/prepare_data.py
!python -m pytest -q

In [ ]:
#@title Ячейка 3 — честный baseline (этап 1): сегментация v11 + Hb по цвету, все датасеты
# ≈10–20 мин на CPU/GPU Colab. Пришлите Claude весь вывод ячейки: таблицу и строки CSV в конце.
import os, pandas as pd
before = len(pd.read_csv('results/metrics_history.csv')) if os.path.exists('results/metrics_history.csv') else 0
!python scripts/evaluate.py --tag baseline_v11
h = pd.read_csv('results/metrics_history.csv')
new = h.iloc[before:]
new.to_csv(f"{os.environ['ANEMIA_DATA']}/../metrics_history_colab.csv", mode='a', index=False,
           header=not os.path.exists(f"{os.environ['ANEMIA_DATA']}/../metrics_history_colab.csv"))
print('\n--- строки для results/metrics_history.csv ---')
print(new.to_csv(index=False))